In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import dask.array as da
import os
import numpy as np
import joblib
import dask  # Import Dask first
dask.config.set({'dataframe.query-planning': False})  # Disable query-planning

import dask.dataframe as dd  # Now import dask.dataframe
from spatialdata import read_zarr

In [14]:

sdata=read_zarr(r"C:\Users\matti\Documents\WERK\STAGE\VIB\data\sdata_channels.zarr")
sdata

c:\Users\matti\.conda\envs\ilastik_napari_182\Lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


SpatialData object, with associated Zarr store: C:\Users\matti\Documents\WERK\STAGE\VIB\data\sdata_channels.zarr
├── Images
│     ├── 'channel_0': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_1': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_2': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_3': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_4': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_5': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_6': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_7': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_8': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_9': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_10': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_11': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_12': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_13': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_14': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_15': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_16': DataArray[cyx]

In [15]:
from enum import StrEnum

class Statistical_Functions(StrEnum):
    SUM = "sum"
    MEAN = "mean"
    COUNT = "count"
    VAR = "var"
    KURTOSIS = "kurtosis"
    SKEW = "skew"
    QUANTILES = "quantiles"
    RADII_AND_AXES_MASK = "axes_mask"

    @staticmethod
    def get_values(args: list["StatisticalFunctions"]) -> list[str]:
        return [stat.value for stat in args]
    
    @staticmethod
    def get_single_stats(stats: list["StatisticalFunctions"]) -> list[str]:
        aggregate_stats = {Statistical_Functions.SUM, 
                    Statistical_Functions.MEAN, 
                    Statistical_Functions.COUNT, 
                    Statistical_Functions.VAR, 
                    Statistical_Functions.KURTOSIS, 
                    Statistical_Functions.SKEW}
        result = []
        for stat in stats:
            if stat in aggregate_stats:
                result.append(stat.value)
        
        return result



In [16]:
def list_tuple_splitter(x):
    a = []
    b = []

    for tup in x:
        a.append(tup[0])
        b.append(tup[1])

    return a, b

f"{prefix} {names[c if isinstance(c, int) else int(c)]}" if c!=Object_Classifier.ID_COLUMN_NAME else c

In [17]:
from functools import reduce
from harpy.utils._aggregate import RasterAggregator
from ilastik.napari.ilastik_exceptions import DepthTooLarge
from ilastik.napari.object_classification import Object_Classifier

In [ ]:
from functools import reduce
from harpy.utils._aggregate import RasterAggregator
from ilastik.napari.ilastik_exceptions import DepthTooLarge
from ilastik.napari.object_classification import Object_Classifier

mask = sdata['masks_whole'].data

images = [('channel_0', sdata['channel_0'].data)]

stats = [Statistical_Functions.QUANTILES, Statistical_Functions.RADII_AND_AXES_MASK]

depth = 100

# feature extraction
mask = mask[None, ...]

names, images = list_tuple_splitter(images)

image = da.concatenate(images)
image=image[ :, None, ... ]

if mask.chunksize != image.chunksize[1:]:
    mask = mask.rechunk(image.chunksize[1:])

features = []

aggregator=RasterAggregator(mask_dask_array=mask, image_dask_array=image)
single_stats = Statistical_Functions.get_single_stats(stats)

if single_stats:
    single_features=aggregator.aggregate_stats(stats_funcs=single_stats)

    for key, feature in zip(single_stats, single_features):
        prefix = key
        feature.columns = [c if c==Object_Classifier.ID_COLUMN_NAME else f"{prefix} {names[c]}" for c in feature.columns]

    features.extend(single_features)

try:
    if Statistical_Functions.QUANTILES in stats:
        quantiles = aggregator.aggregate_quantiles(depth)

        for i in range(len(quantiles)):
            quantile = quantiles[i]
            quantile.columns = [f"{Statistical_Functions.QUANTILES}_{i} {names[c]}" if c!=Object_Classifier.ID_COLUMN_NAME else c for c in quantile.columns]

        features.append(reduce(lambda left, right: dd.merge(left, right, on='cell_ID', how='outer'), quantiles))

    if Statistical_Functions.RADII_AND_AXES_MASK in stats:
        rna = aggregator.aggregate_radii_and_axes(depth)
        rna.columns = [f"{Statistical_Functions.RADII_AND_AXES_MASK}_{c}" if c!=Object_Classifier.ID_COLUMN_NAME else c for c in rna.columns]
        features.append(rna)
except ValueError as e:
    raise DepthTooLarge(e)

res = reduce(lambda left, right: dd.merge(left, right, on=Object_Classifier.ID_COLUMN_NAME, how='outer'), features)
res = res.loc[res[Object_Classifier.ID_COLUMN_NAME]!=0]


Index(['quantiles_0 channel_0', 'cell_ID', 'quantiles_1 channel_0',
       'quantiles_2 channel_0', 'quantiles_3 channel_0',
       'quantiles_4 channel_0', 'quantiles_5 channel_0',
       'quantiles_6 channel_0', 'quantiles_7 channel_0',
       'quantiles_8 channel_0', 'axes_mask_0', 'axes_mask_1', 'axes_mask_2',
       'axes_mask_3', 'axes_mask_4', 'axes_mask_5', 'axes_mask_6',
       'axes_mask_7', 'axes_mask_8', 'axes_mask_9', 'axes_mask_10',
       'axes_mask_11'],
      dtype='object')

In [ ]:
[i for i in Statistical_Functions]

In [ ]:
from harpy.utils._aggregate import RasterAggregator
from functools import reduce
from ilastik.napari.object_classification import check_and_convert_arrays_to_dask

mask = sdata['masks_whole']
images = da.concatenate([sdata['channel_0'],sdata['channel_1']])
annotation = sdata['annotation']
stats = [stat for stat in Statistical_Functions]

mask = check_and_convert_arrays_to_dask(mask)

if len(annotation.shape)>2:
    annotation = annotation.squeeze()
annotation = check_and_convert_arrays_to_dask(annotation)

images = check_and_convert_arrays_to_dask(images)

# start workflow
images=images[ :, None, ... ]

mask = mask[None, ...]

if mask.chunksize != images.chunksize[1:]:
    mask = mask.rechunk(images.chunksize[1:])

aggregator=RasterAggregator(mask_dask_array=mask, image_dask_array=images)

raa = aggregator.aggregate_radii_and_axes(100)

raa

In [ ]:
def feature_extractor(
    self,
    mask: da.Array,
    image: da.Array,
    stats:tuple[Statistical_Functions],
) -> dd.DataFrame:
    # feature extraction
    mask = mask[None, ...]

    if mask.chunksize != image.chunksize[1:]:
        logger.warning("Mask chunks and image chunks are not the same. Changing mask chunks...")
        mask = mask.rechunk(image.chunksize[1:])

    aggregator=RasterAggregator(mask_dask_array=mask, image_dask_array=image)
    single_stats = Statistical_Functions.get_single_stats(stats)
    features=dict(zip(single_stats,aggregator.aggregate_stats(stats_funcs=single_stats)))

    if Statistical_Functions.QUANTILES in stats:
        quantiles = aggregator.aggregate_quantiles(100)

        for i in range(len(quantiles)):
            quantile = quantiles[i]
            quantile.columns = [f"{i}_{c}" if c!='cell_ID' else c for c in quantile.columns]
        
        features[Statistical_Functions.QUANTILES.value] = reduce(lambda left, right: dd.merge(left, right, on='cell_ID', how='outer'), quantiles)

    if Statistical_Functions.RADII_AND_AXES_MASK in stats:
        features[Statistical_Functions.RADII_AND_AXES_MASK.value] = aggregator.aggregate_radii_and_axes(100)


    for key, feature in features.items():
        prefix = key+"_"
        feature.columns = [f"{prefix}{c}" if c!='cell_ID' else c for c in feature.columns]

    res = reduce(lambda left, right: dd.merge(left, right, on='cell_ID', how='outer'), list(features.values()))
    res = res.loc[res['cell_ID']!=0]

    return res


In [ ]:
mask = sdata['masks_whole']
images = sdata['channel_0']
annotation = sdata['annotation']
stats = [stat for stat in Statistical_Functions]
stats

In [ ]:
from ilastik.napari.object_classification import check_and_convert_arrays_to_dask

mask = check_and_convert_arrays_to_dask(mask)

if len(annotation.shape)>2:
    annotation = annotation.squeeze()
annotation = check_and_convert_arrays_to_dask(annotation)

images = check_and_convert_arrays_to_dask(images)

# start workflow
images=images[ :, None, ... ]

features = feature_extractor(mask, images, stats)

features

In [ ]:
features.info()

In [ ]:
from ilastik.napari.utils import get_annotation

annotated_cells_id, annotation=get_annotation( array_1=annotation, array_2=mask)

X_train=features[ features[ "cell_ID" ].isin( annotated_cells_id )]
X_train = X_train.drop("cell_ID", axis=1)

print(type(X_train))
print(type(annotation))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def object_training(
        self,
        X_train: dd.DataFrame,
        y_train: dd.DataFrame,
    ) -> None:
        clf = RandomForestClassifier(n_estimators=100, random_state=42)
        clf.fit(X_train, y_train)

        joblib.dump(clf, os.path.join(self.output_folder, self.MODEL_NAME))

see https://github.com/ilastik/ilastik/blob/54e17482cfe8c186a05b367450c4661792f7048c/ilastik/plugins_default/vigra_objfeats.py#L104 for all features ilastik extracts.

we probably want to support:

image + label:
- 'sum' 
- 'mean'
- 'var'
- 'kurtosis'
- 'skew'
- 'min'
- 'max'
- 'quantiles'

label:
- 'area'
- center_of_mass
- radii and axes

These are all implemented in `RasterAggregator`

In [ ]:
sdata["masks_whole"].data

In [ ]:
import dask.array as da

mask=sdata[ "masks_whole" ].data[ None, ... ] # (z,y,x)

image=da.concatenate([ sdata[ _image_name ].data for _image_name in [*sdata.images] ])
image=image[ :, None, ... ] # ( c,z,y,x )

In [ ]:
print(type(image))

In [ ]:
from harpy.utils._aggregate import RasterAggregator

aggregator=RasterAggregator(mask_dask_array=mask, image_dask_array=image)

In [ ]:
aggregator.aggregate_radii_and_axes( depth=100 ).head()

In [ ]:
quantiles=aggregator.aggregate_quantiles(depth=100 ) # gives you a list of dataframes
quantiles[2].head()

In [ ]:
dfs=aggregator.aggregate_stats( stats_funcs=("sum", "mean", "count", "var", "kurtosis"))

In [ ]:
dfs[0] #-> sum, for each object, and each channel in image

In [ ]:
sdata["annotation"].data

In [ ]:
from ilastik.napari.utils import get_annotation

annotated_cells_id, annotation=get_annotation( array_1=sdata["annotation"].data, array_2=sdata["masks_whole"].data)

print(annotated_cells_id)
print(annotation)

In [ ]:
# for simplicity first try implementing object classification only using mean intensity
features=aggregator.aggregate_stats( stats_funcs=( "mean"  ) )# retuns a list of dataframes, take the first on (mean intensity)
print(len(features))
features[1][ [ 0, 1, "cell_ID" ] ] # only take mean, and only the first two channels
features=features[0][[ 0, 1, "cell_ID" ]]
features=features[  features[ "cell_ID" ]!=0 ] # remove features for background

In [ ]:
def featuer_extractor(mask, image, stats):

    aggregator=RasterAggregator( mask_dask_array=mask, image_dask_array=image)

    features=aggregator.aggregate_stats(stats_funcs=stats)
    for index in range(len(stats)):
        prefix = stats[index]+"_"
        feature = features[index]
        feature.set_index("cell_ID")
        feature.columns = [f"{prefix}{c}" if f"{c}".isdigit() else c for c in feature.columns]

    res = dd.concat(features, axis=1)
    res = res.drop("cell_ID", axis=1)
    res = res.loc[res.index!=0]
    res["cell_ID"] = res.index

    return res

features = featuer_extractor(mask, image, stats=["sum", "mean", "count", "var", "kurtosis", "skew"])

In [ ]:
features

In [ ]:
X_train=features[ features[ "cell_ID" ].isin( annotated_cells_id )]  # train on these
# drop the cell_ID column
X_train=X_train.drop("cell_ID", axis=1)
X_train.head()

In [ ]:
annotation

In [ ]:
annotated_cells_id

In [ ]:
X_train

In [ ]:

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, annotation)
y_pred = clf.predict(X_train)
y_pred

In [ ]:
# run on all data
y_pred_all=clf.predict( features.drop( [ "cell_ID" ], axis=1 ) )  # gives us a prediction for every cell
y_pred_all.shape

In [ ]:
y_pred_all[:10]  # this is a label for every mask

In [ ]:
cell_ids=features[ "cell_ID" ].compute() # cell_IDs
cell_ids
cell_ids[ :10 ]

In [ ]:
# now generate the relabeld mask efficiently
mask=sdata[ "masks_whole" ].data # relabel this

In [ ]:
import dask.array as da

# create the relabeld mask as a dask array

assert cell_ids.shape == y_pred_all.shape

max_id = cell_ids.max()
lookup = np.zeros(max_id + 1, dtype=y_pred_all.dtype)
lookup[cell_ids] = y_pred_all 
relabelled_masks = da.take(lookup, mask) # maps each cell_id to its new label

In [ ]:
from spatialdata.models import Labels2DModel

se= Labels2DModel.parse(relabelled_masks, dims=("y", "x"))

sdata[  "predicted_labels" ] = se

sdata.write(
    r"C:\Users\matti\Documents\WERK\STAGE\VIB\output\object\object_sdata.zarr",
    overwrite=True,
)

sdata = read_zarr(sdata.path)

In [ ]:
from napari_spatialdata import Interactive

Interactive( sdata )

In [ ]:
# dummy code to explain the working of np.take
import numpy as np

lookup = np.array([0, 10, 20, 30, 40])

mask = np.array([
    [3, 1, 2],
    [3, 4, 0]
])

result = np.take(lookup, mask)
result